# Combine Max Raster Values across plans
- Loads the input rasters from a specified folder(s).
- Converts the raster images to arrays
- Creates a max WSEL or depth array by calculating the max of each input rasters per area
- Writes result to a max tif file 

### Import Libraries and define Functions

In [1]:
# %load_ext autoreload
# %autoreload 2

import sys
sys.path.append("..")
from src.raster_functions import *
import json

### Define Directories (specify folder with rasters in this cell)

In [2]:
#Inputs
project = 'wy_fy22'
huc10s = ['1404020005'] 

sub_folder = 'fp_mapping'

In [3]:
root_dir = pl.Path(os.getcwd()).parent
assert root_dir.stem == '_code', 'restart kernel and rerun code'
out_dir = root_dir/'outputs'/project/sub_folder
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

__Note:__ Place all event folders under a model folder within the listed sub_folder.  
Exports from FM will have folders with tif files for each event.

In [ ]:

with open(root_dir/'inputs'/project/'dictionaries'/'completed_event_dictionary.json') as j:
    events_dict = json.load(j)

In [5]:
for huc10 in huc10s:
    huc_dir = root_dir/'inputs'/project/sub_folder/huc10
    
    #get events for each recurrence interval from output data
    #prob_shps = glob.glob(str(huc_dir/'jsons'/'*.geojson'))

    
    ### Define Directories (specify folder with rasters in this cell)
    ### Examine a raster to get resolution info (ensure this cell is updated with proper file-name convention)
    
    # Get a sorted list of tiff files
    list_files_out = glob.glob(str(huc_dir)+'/*/*.tif')
    mapping_type = ['wse','velocity']#'depth'
    for ri, events in events_dict[huc10].items():
        events_rev = [e.replace('-','_') for e in events]
        for map in mapping_type:
            #skip velocities that aren't the 1pct aep per scope
            if map == 'velocity' and ri != '0.01':
                break
            tifs = get_filenames_keyword(list_files_out,events_rev,'or')
            true_tifs = get_filenames_keyword(tifs,[map],'and')
            ### Compute the Max for each overlapping WSEL Pairs
            print('creating max raster for',huc10,ri,map)
            temp, profile = get_tifs_and_profile(huc10,true_tifs)
            create_max_raster_by_block_wy(out_dir,huc10+'_'+map+'_'+ri,true_tifs, profile,force=False)

creating max raster for 1404020005 0.002 wse
creating max raster for 1404020005 0.01m wse
creating max raster for 1404020005 0.01p wse
creating max raster for 1404020005 0.01 wse
creating max raster for 1404020005 0.02 wse
creating max raster for 1404020005 0.04 wse
creating max raster for 1404020005 0.1 wse


### END